In [1]:
"""
의학 텍스트 분류기와 프롬프트를 통합한 파일 (GPT API 형식)
"""
import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
from tenacity import retry, stop_after_attempt, wait_exponential
import re
from prompts.medical_prompts import CCPrompts
from prompts.medical_prompts import TreatmentPrompts
from prompts.medical_prompts import TherapyPrompts
# from prompts.medical_prompts import PresentIllnessPrompts

from prompts.PI_prompts import PresentIllnessPrompts_ver2


from tqdm.asyncio import tqdm as tqdm_asyncio

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_excel('../../data/centum_data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.iloc[:,1:]

api = pd.read_csv('../../data/info.csv')
api_key = api.loc[1][1]

/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_11708/962184981.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  api_key = api.loc[1][1]


In [4]:
df = df[['환자번호', '날짜', 'CC', '약', '장치 ', '습관', '찜질 ', '마사지, 스트레칭', 'PI', 'CMO',
       'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
       'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
       'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', 'End feel', '치료계획',
       'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견']]

In [5]:
df = df[['환자번호', '날짜', 'CC', '약', '장치 ', '습관', '찜질 ', '마사지, 스트레칭', 'PI']]

In [6]:
df = df.sample(100)

In [8]:
df

,환자번호,날짜,CC,약,장치,습관,찜질,"마사지, 스트레칭",PI
3392,2304-09,2023-04-19,초음파ok2008년부터 이갈이때문에 세브란스치과(이갈이장치 치료&체크받다가 안한지1...,NaN,NaN,NaN,NaN,NaN,"12345678 12345678Dr.남윤진료비급여 CT촬영, 파노라마 - K07.6..."
27975,2404-343,2024-05-25,"네이버 검색, 턱관절, 이갈이4월초부터자고 일어나니까 갑자기 오른쪽 턱관절이 입벌릴...",NaN,NaN,NaN,NaN,NaN,Att 많음
5749,2306-219,2023-10-17,"물리치료 , 장치 ck구강내과#3증상: 양쪽 턱 통증 많이 좋아졌어요(vas 6->...",NaN,장치: 3일에 1번 착용./ 불편함 없었어요. 안낀날에는 이를 꽉물고 자게 되는거 같아요,습관: 딱딱하거나 질긴음식 자주 먹었어요./ 치아끼리 안닿도록 노력했어요,찜질: 3일에 1번씩/ 온찜질팩,"마사지,스트레칭: 3일에 1번씩",*#38 부분맹출
20621,2202-153,2022-04-08,"구강내과#4대기고지)물리치료 , 장치 ck통증은 똑같이 없어요.소리는 아침에 나는거...",NaN,NaN,NaN,온찜질:일주일 3번/ 전자렌지핫팩/ 10분,NaN,NaN
26774,2401-383,2024-03-13,"[도착]물리치료 , APS del구강내과#2/한증상: 아침에 일어나면 목이 칼칼하고...",약: 먹고 남았어요/ 불편감 없었어요,NaN,"습관: 딱딱, 질긴음식 피했어요/ 치아끼리 안닿게 턱에 힘 풀려고 하는데 무의식적으...",찜질: 못했어요,"마사지,스트레칭: 마사지 주3회, 스트레칭 주3회",NaN
...,...,...,...,...,...,...,...,...,...
18296,2112-103,2022-01-04,[도착]인레이 셋팅임시재료부분에 찌릿한느낌 씹을 때 눌리는느낌약 일주일치먹고 부작용...,NaN,NaN,NaN,NaN,NaN,NaN
20247,2201-53,2022-01-13,3년전부터 턱이 1년에 2번 스케일링하는데 간혹 빠질때가있어요.평소에는 입을 오래 ...,NaN,NaN,NaN,NaN,NaN,"12345678 12345678Dr.남윤진료측두하악장애분석검사, 파노라마, CT촬영..."
26282,2206-92,2022-11-05,"물리치료 , 증상 ck증상: 통증 없는채로 유지중 / 하품할때만 오른쪽 불편감과 소...",NaN,NaN,NaN,온찜질: 잘 못했어요 . 증상 없어서 안하게 돼요,NaN,47번 협면 C212345678 12345678Dr.남윤진료TMJ자극요법-단순 - ...
22506,2204-103,2022-04-25,"구강내과#2[도착]물리치료 , APS del증상 : 입 벌리는 것도, 가만히 있을 ...",NaN,NaN,NaN,NaN,NaN,치아 교모 있음44번 CA


In [14]:
class Config:
    # 실제 OpenAI API 키로 교체하거나 환경변수로부터 로드하세요.
    API_KEY = api_key
    MODEL_NAME = "gpt-4o"
    MAX_TOKENS = 4096
    TEMPERATURE = 0
    BATCH_SIZE = 50
    SEMAPHORE_LIMIT = 5
    MAX_RETRIES = 3
    CHECKPOINT_DIR = "checkpoints"
    LOG_FILE = "medical_classifier.log"


#############################################
# 로깅 설정 함수
#############################################

def setup_logging(log_file=Config.LOG_FILE):
    """로깅 설정을 초기화하는 함수"""
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

# 초기 로거 생성 (설정은 아직 적용되지 않음)
logger = logging.getLogger(__name__)


#############################################
# 체크포인트 관리 클래스
#############################################

class CheckpointManager:
    """Checkpoint management class"""
    def __init__(self, checkpoint_dir: str = Config.CHECKPOINT_DIR):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(self.checkpoint_dir, exist_ok=True)
    
    def get_checkpoint_path(self, column: str) -> str:
        safe_column = column.replace("/", "_").replace("\\", "_")
        return os.path.join(self.checkpoint_dir, f"{safe_column}_checkpoint.parquet")

    def save_checkpoint(self, df: pd.DataFrame, column: str) -> None:
        try:
            df.to_parquet(self.get_checkpoint_path(column))
            logger.info(f"Checkpoint saved for column {column}")
        except Exception as e:
            logger.error(f"Failed to save checkpoint for {column}: {str(e)}")

    def load_checkpoint(self, column: str) -> Optional[pd.DataFrame]:
        path = self.get_checkpoint_path(column)
        if os.path.exists(path):
            try:
                return pd.read_parquet(path)
            except Exception as e:
                logger.error(f"Failed to load checkpoint for {column}: {str(e)}")
        return None


#############################################
# 메디컬 텍스트 분류기 클래스 (GPT API 사용)
#############################################

class MedicalTextClassifier:
    """의학 텍스트 분류기 클래스"""
    
    def __init__(self, api_key: str, config=None):
        """초기화"""
        self.config = config if config is not None else Config
        self.client = openai.OpenAI(api_key=api_key)  # 클라이언트 객체 생성 방식 변경
        self.semaphore = asyncio.Semaphore(self.config.SEMAPHORE_LIMIT)
        self.checkpoint = CheckpointManager(self.config.CHECKPOINT_DIR)
        
        # 분류기 메소드 맵핑
        self.classifiers = {
            # 'CC': self._classify_cc,
            # '약': self._classify_medication,
            # '장치': self._classify_device,
            # '습관': self._classify_habit,
            # '찜질': self._classify_hot_pack,
            # '마사지, 스트레칭': self._classify_massage,
            'PI': self._classify_present_illness,
            # '물리치료': self._classify_physical_therapy,
        }
    
    async def process_all_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """모든 컬럼 처리"""
        original_shape = df.shape
        processed_cols = 0
        
        for column in self.classifiers.keys():
            if column in df.columns:
                logger.info(f"Processing column: {column}")
                df = await self._process_column_with_checkpoint(df, column)
                
                # 처리 후 해당 컬럼에서 파생된 새 컬럼 확인
                derived_cols = [col for col in df.columns if col.startswith(f"{column}_")]
                logger.info(f"Column {column} generated {len(derived_cols)} derived columns: {derived_cols}")
                
                # 파생 컬럼의 값이 있는 행 수 확인
                for derived_col in derived_cols:
                    non_empty_count = df[derived_col].notna().sum()
                    logger.info(f"Column {derived_col} has {non_empty_count} non-empty values")
                
                processed_cols += 1
        
        logger.info(f"Original DataFrame shape: {original_shape}, Processed columns: {processed_cols}")
        return df

    async def _process_column_with_checkpoint(self, df: pd.DataFrame, column: str) -> pd.DataFrame:
        """체크포인트를 사용한 컬럼 처리"""
        try:
            # 기존 체크포인트 확인
            checkpoint_df = self.checkpoint.load_checkpoint(column)
            if checkpoint_df is not None:
                # 체크포인트로부터 원본 데이터프레임 업데이트
                for idx in checkpoint_df.index:
                    if idx in df.index:
                        for col in checkpoint_df.columns:
                            # 원본 DataFrame에 직접 값 설정
                            df.loc[idx, col] = checkpoint_df.loc[idx, col]
                logger.info(f"Resumed from checkpoint for {column}")
                return df

            # 유효한 텍스트 처리
            mask = df[column].notna() & df[column].str.strip().astype(bool)
            if not mask.any():
                return df

            texts_with_idx = [(idx, text) for idx, text in df.loc[mask, column].items()]
            
            # 로깅 추가: 처리 대상 텍스트 출력
            for idx, text in texts_with_idx:
                logger.info(f"Processing text at index {idx}, first 100 chars: {text[:100]}")
            
            # 데이터 처리 - 여기서 results 변수가 정의됨
            results = await self._safe_process_batches(
                texts=[text for _, text in texts_with_idx],
                original_indices=[idx for idx, _ in texts_with_idx],
                classifier=self.classifiers[column],
                column=column
            )

            # 결과 처리
            if results:
                result_df = pd.DataFrame(results).set_index('index')
                logger.info(f"Result DataFrame shape: {result_df.shape}, indices: {result_df.index.tolist()}")
                
                for col in result_df.columns:
                    new_col = f"{column}_{col}"
                    
                    # 각 인덱스에 명시적으로 할당
                    for idx in result_df.index:
                        if idx in df.index:
                            logger.debug(f"Setting value at index {idx}, column {new_col}: {result_df.loc[idx, col]}")
                            df.loc[idx, new_col] = result_df.loc[idx, col]
                        else:
                            logger.warning(f"Index {idx} from result not found in DataFrame")

            self._cleanup_checkpoint(column)
            return df

        except Exception as e:
            logger.error(f"Critical error processing {column}: {str(e)}")
            raise

    async def _safe_process_batches(self, texts: List[str], original_indices: List[int],
                                classifier, column: str) -> List[Dict]:
        """더 높은 신뢰성을 위해 한 번에 하나의 레코드 처리"""
        results = []

        for i, (text, idx) in enumerate(zip(texts, original_indices)):
            try:
                logger.info(f"단일 텍스트 처리 중 {i+1}/{len(texts)} (인덱스 {idx})")
                
                # 하나의 텍스트만 처리
                batch_result = await self._process_with_retry(classifier, [text], [idx])
                
                if batch_result:
                    results.extend(batch_result)
                    
                    # 이 결과만으로 DataFrame 생성
                    partial_df = pd.DataFrame(batch_result).set_index('index')
                    self.checkpoint.save_checkpoint(partial_df, column)
                    
            except Exception as e:
                logger.error(f"인덱스 {idx}의 텍스트 처리 실패: {str(e)}")
                # 인덱스 매핑을 유지하기 위해 빈 결과 추가
                results.append({"index": idx})

        return results

    @retry(stop=stop_after_attempt(3),
           wait=wait_exponential(multiplier=1, min=2, max=10))
# _process_with_retry 메서드에 추가
    async def _process_with_retry(self, classifier, batch_texts: List[str], batch_indices: List[int]) -> List[Dict]:
        """각 텍스트를 개별적으로 처리하여 정확한 매핑 보장"""
        results = []

        for i, text in enumerate(batch_texts):
            async with self.semaphore:
                # 한 번에 하나의 텍스트 처리
                single_result = await classifier([text], self.semaphore)
                if single_result and len(single_result) > 0:
                    # 결과에 인덱스 추가
                    results.append({"index": batch_indices[i], **single_result[0]})
                else:
                    # API가 빈 또는 유효하지 않은 결과를 반환한 경우 처리
                    results.append({"index": batch_indices[i]})
                
                # 각 개별 결과 로깅
                logger.info(f"텍스트 {i} (인덱스 {batch_indices[i]}) 처리 완료: {single_result[0] if single_result and len(single_result) > 0 else '결과 없음'}")
        
        return results

    def _cleanup_checkpoint(self, column: str) -> None:
        """성공적인 처리 후 체크포인트 정리"""
        try:
            checkpoint_path = self.checkpoint.get_checkpoint_path(column)
            if os.path.exists(checkpoint_path):
                os.remove(checkpoint_path)
                logger.info(f"Checkpoint cleaned up for {column}")
        except Exception as e:
            logger.error(f"Failed to cleanup checkpoint for {column}: {str(e)}")

    @retry(stop=stop_after_attempt(3),
        wait=wait_exponential(multiplier=1, min=2, max=10))
    async def _make_api_call(self, prompt: str, semaphore: asyncio.Semaphore) -> List[Dict]:
        """API 호출 메소드 (OpenAI API v1.0.0+)"""
        try:
            async with semaphore:
                client = openai.OpenAI(api_key=self.config.API_KEY)
                response = await asyncio.to_thread(
                    client.chat.completions.create,
                    model=self.config.MODEL_NAME,
                    messages=[
                        {"role": "system", "content": "JSON 형식으로 응답하세요."},
                        {"role": "user", "content": prompt}
                    ],
                    max_tokens=self.config.MAX_TOKENS,
                    temperature=self.config.TEMPERATURE
                )
                content = response.choices[0].message.content
                logger.debug(f"API Response: {content[:200]}...")
                result = self._validate_and_parse_json(content)
                if not result:
                    raise ValueError("Invalid JSON structure")
                return result

        except Exception as e:
            logger.error(f"API call failed: {str(e)}")
            raise

    def _validate_and_parse_json(self, content: str) -> List[Dict]:
        """향상된 JSON 응답 검증 및 파싱"""
        try:
            # 원시 내용 로깅
            logger.info(f"원시 API 응답: {content[:500]}...")
            
            # 먼저 직접 JSON 파싱 시도
            try:
                parsed = json.loads(content)
                if isinstance(parsed, list):
                    return parsed
            except json.JSONDecodeError:
                pass  # 직접 파싱이 실패하면 정규식 추출로 계속 진행
            
            # 정규식으로 JSON 추출
            json_pattern = r'```json\s*([\s\S]*?)\s*```|(\[[\s\S]*\])'
            matches = re.findall(json_pattern, content)
            
            for match in matches:
                # 각 일치 항목 시도
                for m in match:
                    if not m.strip():
                        continue
                        
                    try:
                        parsed = json.loads(m.strip())
                        if isinstance(parsed, list):
                            logger.info(f"JSON 파싱 성공: {parsed}")
                            return parsed
                    except:
                        continue
            
            # 마지막 수단: JSON 객체나 배열처럼 보이는 것 찾기
            fallback_pattern = r'(\{[\s\S]*?\}|\[[\s\S]*?\])'
            fallback_matches = re.findall(fallback_pattern, content)
            
            for m in fallback_matches:
                try:
                    parsed = json.loads(m.strip())
                    if isinstance(parsed, dict):
                        # 단일 객체를 목록으로 변환
                        return [parsed]
                    elif isinstance(parsed, list):
                        return parsed
                except:
                    continue
                    
            logger.error(f"응답에서 JSON을 파싱하지 못함")
            return []
            
        except Exception as e:
            logger.error(f"JSON 파싱 오류: {str(e)}")
            return []
        
    async def _classify_cc(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """Chief Complaints 분류"""
        # CC를 3개의 작은 프롬프트로 분할
        cc_results = await self._make_api_call(CCPrompts.cc_analysis_prompt(texts), semaphore)
        history_results = await self._make_api_call(CCPrompts.cc_history_prompt(texts), semaphore)
        severity_results = await self._make_api_call(CCPrompts.cc_severity_prompt(texts), semaphore)
        
        # 결과 병합
        combined_results = []
        for i in range(len(texts)):
            combined_dict = {}
            if i < len(cc_results):
                combined_dict.update(cc_results[i])
            if i < len(history_results):
                combined_dict.update(history_results[i])
            if i < len(severity_results):
                combined_dict.update(severity_results[i])
            
            combined_results.append(combined_dict)
        
        return combined_results
    
    async def _classify_medication(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """약물 복용 분류"""
        results = await self._make_api_call(TreatmentPrompts.medication_prompt(texts), semaphore)
        logger.debug(f"약물 분류 결과: {results}")
        return results

    async def _classify_device(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """장치 사용 분류"""
        results = await self._make_api_call(TreatmentPrompts.device_prompt(texts), semaphore)
        logger.debug(f"장치 분류 결과: {results}")
        return results

    async def _classify_habit(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """습관 분류"""
        results = await self._make_api_call(TreatmentPrompts.habit_prompt(texts), semaphore)
        logger.debug(f"습관 분류 결과: {results}")
        return results

    async def _classify_hot_pack(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """찜질 분류"""
        results = await self._make_api_call(TherapyPrompts.hot_pack_prompt(texts), semaphore)
        logger.debug(f"찜질 분류 결과: {results}")
        return results
    
    async def _classify_massage(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """마사지 및 스트레칭 분류"""
        results = await self._make_api_call(TherapyPrompts.massage_prompt(texts), semaphore)
        logger.debug(f"마사지 분류 결과: {results}")
        return results 
    
    # async def _classify_present_illness(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
    #     """PI 분류 개선"""
    #     limited_texts = [text[:3000] if text and len(text) > 3000 else text for text in texts]
    #     smaller_batch_size = 10  # PI에 대해 더 작은 배치 크기 사용
    #     all_results = []
        
    #     for i in range(0, len(limited_texts), smaller_batch_size):
    #         batch = limited_texts[i:i+smaller_batch_size]
    #         try:
    #             logger.info(f"Processing PI basic info batch {i//smaller_batch_size + 1}")
    #             pi_basic_results = await self._make_api_call(PresentIllnessPrompts.pi_basic_info_prompt(batch), semaphore)
                
    #             logger.info(f"Processing PI examination batch {i//smaller_batch_size + 1}")
    #             pi_exam_results = await self._make_api_call(PresentIllnessPrompts.pi_examination_prompt(batch), semaphore)
                
    #             logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
    #             pi_treatment_results = await self._make_api_call(PresentIllnessPrompts.pi_treatment_details_prompt(batch), semaphore)
                
    #             for j in range(len(batch)):
    #                 combined_dict = {}
    #                 if j < len(pi_basic_results):
    #                     combined_dict.update(pi_basic_results[j])
    #                 if j < len(pi_exam_results):
    #                     combined_dict.update(pi_exam_results[j])
    #                 if j < len(pi_treatment_results):
    #                     combined_dict.update(pi_treatment_results[j])
    #                 all_results.append(combined_dict)
            
    #         except Exception as e:
    #             logger.error(f"Error processing PI batch {i//smaller_batch_size + 1}: {str(e)}")
    #             all_results.extend([{} for _ in range(len(batch))])
        
    #     while len(all_results) < len(texts):
    #         all_results.append({})
        
    #     return all_results[:len(texts)]

        
    async def _classify_present_illness(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:

        """PI 분류 개선"""
        limited_texts = [text[:10000] if text and len(text) > 10000 else text for text in texts]
        
        # 로깅 추가: 제한된 텍스트 확인
        for i, text in enumerate(limited_texts):
            logger.info(f"Limited text {i}, length {len(text)}, first 100 chars: {text[:100]}")
        
        smaller_batch_size = 10  # PI에 대해 더 작은 배치 크기 사용
        all_results = []
        
        for i in range(0, len(limited_texts), smaller_batch_size):
            batch = limited_texts[i:i+smaller_batch_size]
            batch_results = []  # 각 배치의 결과를 저장할 리스트
            
            # 로깅 추가: 배치 크기 및 첫 번째 텍스트 확인
            logger.info(f"Processing batch {i//smaller_batch_size + 1}, size: {len(batch)}")
            if batch:
                logger.info(f"First item in batch, length {len(batch[0])}, first 100 chars: {batch[0][:100]}")
        
            try:
                logger.info(f"Processing PI basic info batch {i//smaller_batch_size + 1}")
                pi_aggravating_factors_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_aggravating_factors(batch), semaphore)
                
                logger.info(f"Processing PI examination batch {i//smaller_batch_size + 1}")
                pi_TMJ_PI_desc_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_TMJ_PI_desc(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_TMJ_PI_treatment_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_TMJ_PI_treatment(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_drug_treatment_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_drug_treatment(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_closing_dentalgear_desc_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_closing_dentalgear_desc(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_PI_check_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_PI_check(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_PI_diagnosis_jojint_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_PI_diagnosis_jojint(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_occlusal_treatment_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_occlusal_treatment(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_medication_prescription_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_medication_prescription(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_other_treatment_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_other_treatment(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_onset_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_onset(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_pattern_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_pattern(batch), semaphore)
                
                logger.info(f"Processing PI treatment batch {i//smaller_batch_size + 1}")
                pi_status_results = await self._make_api_call(PresentIllnessPrompts_ver2.pi_status(batch), semaphore)
                
                for j in range(len(batch)):
                    combined_dict = {}
                    if j < len(pi_aggravating_factors_results):
                        combined_dict.update(pi_aggravating_factors_results[j])
                    if j < len(pi_TMJ_PI_desc_results):
                        combined_dict.update(pi_TMJ_PI_desc_results[j])
                    if j < len(pi_TMJ_PI_treatment_results):
                        combined_dict.update(pi_TMJ_PI_treatment_results[j])
                    if j < len(pi_drug_treatment_results):  # pi_drug_treatment
                        combined_dict.update(pi_drug_treatment_results[j])
                    if j < len(pi_closing_dentalgear_desc_results):  # pi_closing_dentalgear_desc
                        combined_dict.update(pi_closing_dentalgear_desc_results[j])
                    if j < len(pi_PI_check_results):  # pi_PI_check
                        combined_dict.update(pi_PI_check_results[j])
                    if j < len(pi_PI_diagnosis_jojint_results):  # pi_PI_diagnosis_joint
                        combined_dict.update(pi_PI_diagnosis_jojint_results[j])
                    if j < len(pi_occlusal_treatment_results):  # pi_occlusal_treatment
                        combined_dict.update(pi_occlusal_treatment_results[j])
                    if j < len(pi_medication_prescription_results):  # pi_medication_prescription
                        combined_dict.update(pi_medication_prescription_results[j])
                    if j < len(pi_other_treatment_results):  # pi_other_treatment
                        combined_dict.update(pi_other_treatment_results[j])
                    if j < len(pi_onset_results):  # pi_onset
                        combined_dict.update(pi_onset_results[j])
                    if j < len(pi_pattern_results):  # pi_pattern
                        combined_dict.update(pi_pattern_results[j])
                    if j < len(pi_status_results):  # pi_status
                        combined_dict.update(pi_status_results[j])
                    
                    batch_results.append(combined_dict)
                
                all_results.extend(batch_results)
            
            except Exception as e:
                logger.error(f"Error processing PI batch {i//smaller_batch_size + 1}: {str(e)}")
                all_results.extend([{} for _ in range(len(batch))])
        
        while len(all_results) < len(texts):
            all_results.append({})
        
        return all_results[:len(texts)]

        
#############################################
# 의학 데이터 처리 함수
#############################################

async def process_medical_data(df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """Process medical data with comprehensive error handling and logging"""
    classifier = MedicalTextClassifier(api_key)
    start_time = datetime.now()
    logger.info(f"Starting medical data processing at {start_time}")

    try:
        processed_df = await classifier.process_all_columns(df)

        end_time = datetime.now()
        processing_time = end_time - start_time
        total_rows = len(df)
        processed_columns = [col for col in df.columns if col in classifier.classifiers]

        logger.info("=== Processing Summary ===")
        logger.info(f"Total time: {processing_time}")
        logger.info(f"Total rows processed: {total_rows}")
        logger.info(f"Columns processed: {processed_columns}")

        for col in processed_columns:
            total_entries = df[col].notna().sum()
            processed_entries = sum(1 for col_name in processed_df.columns
                                 if col_name.startswith(f"{col}_")
                                 and processed_df[col_name].notna().any())
            success_rate = (processed_entries / total_entries * 100) if total_entries > 0 else 0
            logger.info(f"{col} - Success rate: {success_rate:.2f}%")

        return processed_df

    except Exception as e:
        logger.critical(f"Critical error during medical data processing: {str(e)}")
        raise


#############################################
# 메인 함수
#############################################

def main():
    """메인 함수"""
    global logger
    logger = setup_logging()

    try:
        # 실제 환자 데이터를 로드하는 경우 아래 주석을 해제하세요.
        # df = pd.read_excel('환자데이터.xlsx')
        
        # 테스트용 샘플 데이터 생성
        df_sample = df.head(10)
        
        api_key = Config.API_KEY
        
        logger.info("Starting sample data processing")
        loop = asyncio.get_event_loop()
        processed_df = loop.run_until_complete(process_medical_data(df_sample, api_key))
        
        pi_cols = [col for col in processed_df.columns if col.startswith('PI_')]
        logger.info(f"Total PI derived columns: {len(pi_cols)}")
        
        if len(pi_cols) > 0:
            for col in pi_cols:
                non_empty = processed_df[col].notna().sum()
                logger.info(f"Column {col}: {non_empty} non-empty values")
        else:
            logger.warning("No PI derived columns found in processed DataFrame!")
            
        # 타입 충돌 해결을 위한 열 타입 변환
        for col in processed_df.columns:
            if col.endswith('_pi_next_schedule'):
                # 빈 문자열을 NaN으로 변환 후 정수형으로 변환
                processed_df[col] = pd.to_numeric(processed_df[col].replace('', pd.NA), errors='coerce')
            elif processed_df[col].dtype == 'object':
                # 다른 객체 타입의 열은 모두 문자열로 유지
                processed_df[col] = processed_df[col].astype(str)
                
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = f'processed_medical_data_{timestamp}.parquet'
        processed_df.to_parquet(output_file)
        logger.info(f"Data successfully saved to {output_file}")
        
        logger.info("\n=== Processing Results ===")
        for column in processed_df.columns:
            if '_' in column:  # 파생 컬럼만 표시
                valid_count = processed_df[column].notna().sum()
                logger.info(f"{column}: {valid_count} valid entries")
                if processed_df[column].dtype in ['object', 'category']:
                    value_counts = processed_df[column].value_counts()
                    logger.info(f"Value distribution:\n{value_counts}\n")
                    
    except Exception as e:
        logger.error(f"Main execution failed: {str(e)}")
        sys.exit(1)
    finally:
        logger.info("Program execution completed")


if __name__ == "__main__":
    main()

2025-03-03 00:14:18,745 - __main__ - INFO - Starting sample data processing
2025-03-03 00:14:18,759 - __main__ - INFO - Starting medical data processing at 2025-03-03 00:14:18.759523
2025-03-03 00:14:18,761 - __main__ - INFO - Processing column: PI
2025-03-03 00:14:18,769 - __main__ - INFO - Processing text at index 3392, first 100 chars: 12345678 12345678Dr.남윤진료비급여 CT촬영, 파노라마 - K07.65 턱관절의 퇴행성관절염- 파노라마- 파노라마(특수)- 측두하악장애분석검사- CT 촬영(비급여)파
2025-03-03 00:14:18,769 - __main__ - INFO - Processing text at index 27975, first 100 chars: Att 많음
2025-03-03 00:14:18,769 - __main__ - INFO - Processing text at index 5749, first 100 chars: *#38 부분맹출
2025-03-03 00:14:18,770 - __main__ - INFO - Processing text at index 27084, first 100 chars: 12345678 12345678Dr.남윤진료측두하악장애분석검사, 파노라마, CT촬영 - K07.65 턱관절의 퇴행성관절염- 파노라마- 파노라마(특수)- Cone Beam CT- 측
2025-03-03 00:14:18,770 - __main__ - INFO - 단일 텍스트 처리 중 1/4 (인덱스 3392)
2025-03-03 00:14:18,770 - __main__ - INFO - Limited text 0, length 314, first 100 chars: 12

In [15]:
df.head(1)

,환자번호,날짜,CC,약,장치,습관,찜질,"마사지, 스트레칭",PI
14424,2211-238,2024-04-27,"물리치료 , 장치 ck, x-ray구강내과#17 / 한증상: 입 잘 벌어져요. 턱...",NaN,"장치: 주 3회 착용 (APS 2회, SS1회)/ 밴드X/ 불편감X","습관: 딱딱, 질긴음식 안먹어요/ 치아끼리 안닿게 턱에 힘 풀어줘요",찜질: 온수찜질 매일 5분 정도,"마사지,스트레칭: 매일 해요","판독#18,28 - 악관절의 퇴행성 관절염 [Dr.남윤] - 서명일치"


In [15]:
pd.read_parquet('processed_medical_data_20250303_001538.parquet')

,환자번호,날짜,CC,약,장치,습관,찜질,"마사지, 스트레칭",PI,PI_pi_aggravating_factors,PI_pi_TMJ_PI_desc,PI_pi_TMJ_PI_treatment,PI_pi_drug_treatment,PI_pi_closing_dentalgear_desc,PI_pi_next_schedule,PI_pi_next_ck,PI_pi_PI_diagnosis_jojint,PI_pi_occlusal_treatment,PI_pi_medication_prescription,PI_pi_other_treatment,PI_pi_onset,PI_pi_pattern,PI_pi_status
3392,2304-09,2023-04-19,초음파ok2008년부터 이갈이때문에 세브란스치과(이갈이장치 치료&체크받다가 안한지1...,nan,nan,nan,nan,nan,"12345678 12345678Dr.남윤진료비급여 CT촬영, 파노라마 - K07.6...",,"파노라마, 파노라마(특수), 측두하악장애분석검사, 초음파",물리치료,,SS,NaN,물리치료,퇴행성 관절염 (K07.65),SS Splint 인상채득,,,,,
27975,2404-343,2024-05-25,"네이버 검색, 턱관절, 이갈이4월초부터자고 일어나니까 갑자기 오른쪽 턱관절이 입벌릴...",nan,nan,nan,nan,nan,Att 많음,,,,,,NaN,,,,,,,,
5749,2306-219,2023-10-17,"물리치료 , 장치 ck구강내과#3증상: 양쪽 턱 통증 많이 좋아졌어요(vas 6->...",nan,장치: 3일에 1번 착용./ 불편함 없었어요. 안낀날에는 이를 꽉물고 자게 되는거 같아요,습관: 딱딱하거나 질긴음식 자주 먹었어요./ 치아끼리 안닿도록 노력했어요,찜질: 3일에 1번씩/ 온찜질팩,"마사지,스트레칭: 3일에 1번씩",*#38 부분맹출,,,,,,NaN,,,,,,,,
20621,2202-153,2022-04-08,"구강내과#4대기고지)물리치료 , 장치 ck통증은 똑같이 없어요.소리는 아침에 나는거...",nan,nan,nan,온찜질:일주일 3번/ 전자렌지핫팩/ 10분,nan,nan,nan,nan,nan,nan,nan,NaN,nan,nan,nan,nan,nan,nan,nan,nan
26774,2401-383,2024-03-13,"[도착]물리치료 , APS del구강내과#2/한증상: 아침에 일어나면 목이 칼칼하고...",약: 먹고 남았어요/ 불편감 없었어요,nan,"습관: 딱딱, 질긴음식 피했어요/ 치아끼리 안닿게 턱에 힘 풀려고 하는데 무의식적으...",찜질: 못했어요,"마사지,스트레칭: 마사지 주3회, 스트레칭 주3회",nan,nan,nan,nan,nan,nan,NaN,nan,nan,nan,nan,nan,nan,nan,nan
27084,2402-247,2024-03-04,"통증이 너무 심하여 턱관절 디스크 상황 알고싶습니다 / 추천받았습니다, 턱관절, 기...",nan,nan,nan,nan,nan,"12345678 12345678Dr.남윤진료측두하악장애분석검사, 파노라마, CT촬영...",,"측두하악장애분석검사, 파노라마, Cone Beam CT, 초음파, 파노라마(특수)","분사신장치료, 악관절 고착 해소술, 측두하악관절자극요법-단순자극, 측두하악관절자극요...","페리슨정(에페리손염산염), 세크로정(아세클로페낙), 휴모리드정5mg(모사프리드시트르...",APS,14.0,물리치료,"퇴행성 관절염 (K07.65), 턱관절 통증 (K07.63), 저작근의 장애 (K0...",APS Splint 인상채득,"페리슨정(에페리손염산염) - 1/1회/5일, 세크로정(아세클로페낙) - 1/2회/5...",,,,
18146,2111-64,2023-02-22,CT찍고 기존 장치 가지고 오시면 CK 좋아지진않은 것 같아요하악 : SS(트랙션 ...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,NaN,nan,nan,nan,nan,nan,nan,nan,nan
794,2301-357,2023-02-14,물리치료만,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,NaN,nan,nan,nan,nan,nan,nan,nan,nan
25120,2206-11,2023-03-02,"구강내과#12I.O찍고 보철물 교체CK, 물리치료 , 장치 ck, x-ray증상: ...",nan,장치: 격일로 끼라고 하셔서 그렇게 꼈어요. 장치도 이제 너무 편해졌어요,습관: 치아끼리 닿지 않게 하려고 했어요,찜질: 매일 했어요,nan,nan,nan,nan,nan,nan,nan,NaN,nan,nan,nan,nan,nan,nan,nan,nan
24004,2205-177,2023-06-30,"구강내과#7물리치료 , 증상 ck, x-ray증상: 전에 턱 뻐근한거 있다가 사라졌...",nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,NaN,nan,nan,nan,nan,nan,nan,nan,nan


In [50]:
# 컬럼명 자세히 확인
for col in df.columns:
    print(f"컬럼: '{col}', 길이: {len(col)}")

컬럼: '환자번호', 길이: 4
컬럼: '날짜', 길이: 2
컬럼: 'CC', 길이: 2
컬럼: '약', 길이: 1
컬럼: '장치 ', 길이: 3
컬럼: '습관', 길이: 2
컬럼: '찜질 ', 길이: 3
컬럼: '마사지, 스트레칭', 길이: 9
컬럼: 'PI', 길이: 2
컬럼: 'CMO', 길이: 3
컬럼: 'MMO', 길이: 3
컬럼: 'Cap.pal', 길이: 7
컬럼: 'M.pal', 길이: 5
컬럼: 'Noise', 길이: 5
컬럼: 'Loading', 길이: 7
컬럼: 'Occlusion', 길이: 9
컬럼: 'OJ/OB', 길이: 5
컬럼: 'Class', 길이: 5
컬럼: 'Midline Shift', 길이: 13
컬럼: 'Deviation', 길이: 9
컬럼: 'CR-CO', 길이: 5
컬럼: 'Tongue ridging', 길이: 14
컬럼: 'Mucosal ridging', 길이: 15
컬럼: 'Ultrasono', 길이: 9
컬럼: 'Rt', 길이: 2
컬럼: 'Lt', 길이: 2
컬럼: 'End feel', 길이: 8
컬럼: '치료계획', 길이: 4
컬럼: 'T-scan 악화/개선', 길이: 12
컬럼: 'CBCT 악화/개선', 길이: 10
컬럼: 'CBCT 판독소견', 길이: 9


In [6]:
print("Available methods in PresentIllnessPrompts_ver2:", [method for method in dir(PresentIllnessPrompts_ver2) if method.startswith('pi_')])

Available methods in PresentIllnessPrompts_ver2: ['pi_PI_check', 'pi_PI_diagnosis_jojint', 'pi_TMJ_PI_desc', 'pi_TMJ_PI_treatment', 'pi_aggravating_factors', 'pi_closing_dentalgear_desc', 'pi_drug_treatment', 'pi_medication_prescription', 'pi_occlusal_treatment', 'pi_onset', 'pi_other_treatment', 'pi_pattern', 'pi_physical_therapy', 'pi_status']
